# Experiment 0 / 0.5 Colab Runbook

This notebook runs the thesis prototype pipeline on Google Colab GPU.

**Scope**
- Experiment 0: second-stage patient/candidate reranking.
- Experiment 0.5: detector diagnostic branch for high-recall candidate generation.
- Real evidence starts when you run patient-disjoint SLICE-3D and later iToBoS.

**Before running**
1. Push your repo to GitHub.
2. Accept dataset rules on Kaggle/ISIC for any datasets you use.
3. Add `kaggle.json` when prompted or mount Google Drive where it is stored.

## 0. GPU and environment check

In [ ]:
import os, sys, subprocess, json, textwrap, pathlib, glob, shutil
print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed yet:", e)

!nvidia-smi || true

## 1. Clone your GitHub repo

In [ ]:
# CHANGE THIS after pushing to GitHub
GITHUB_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
BRANCH = "main"

WORKDIR = "/content/thesis_experiment"
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

# If you have not pushed yet, upload the .tar.gz manually and extract it instead.
if "YOUR_USERNAME" not in GITHUB_URL:
    !git clone --branch {BRANCH} {GITHUB_URL} {WORKDIR}
else:
    print("Set GITHUB_URL first. For now, this cell will not clone anything.")
    print("Alternative: upload experiment0_debugged.tar.gz to Colab and extract it:")
    print("  from google.colab import files; files.upload()")

## 1B. Alternative: upload the tar.gz package instead of cloning

In [ ]:
# Use this if you have not pushed to GitHub yet.
# from google.colab import files
# uploaded = files.upload()
# Example expected file: experiment0_debugged.tar.gz or experiment0_with_detector.tar.gz

import tarfile, os, glob, shutil, pathlib

uploaded_archives = glob.glob("/content/*.tar.gz")
if uploaded_archives and not os.path.exists(WORKDIR):
    archive = uploaded_archives[0]
    print("Extracting:", archive)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall("/content")
    # Most packages extract to /content/experiment0
    if os.path.exists("/content/experiment0"):
        shutil.move("/content/experiment0", WORKDIR)
        print("Moved to", WORKDIR)
else:
    print("No uploaded tar.gz found or WORKDIR already exists.")

## 2. Install dependencies

In [ ]:
%cd {WORKDIR} 2>/dev/null || true

# Basic install. If requirements.txt is heavy, install progressively.
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
else:
    !pip install -q torch torchvision torchaudio pandas numpy scikit-learn matplotlib seaborn pyyaml tqdm pillow opencv-python pycocotools pytest

# Optional packages for visualization and Kaggle downloads
!pip install -q kaggle umap-learn h5py ipywidgets

print("Installed.")

## 3. Run unit tests and toy Experiment 0

In [ ]:
%cd {WORKDIR}

# Unit tests should pass before using real data.
!python -m pytest tests/ -v || true

# Run toy reranking experiment.
!python run_experiment0.py

# Show key outputs.
import os, json, glob
print("Outputs:", glob.glob("outputs/*")[:20])
if os.path.exists("outputs/metrics.json"):
    with open("outputs/metrics.json") as f:
        metrics = json.load(f)
    print("Rerankers:", list(metrics.keys())[:10])

## 4. Kaggle setup

In [ ]:
# Option A: upload kaggle.json manually.
# from google.colab import files
# files.upload()  # upload kaggle.json

# Option B: keep kaggle.json in Google Drive and copy it.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p ~/.kaggle
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json

# If you uploaded kaggle.json into /content:
if os.path.exists("/content/kaggle.json"):
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json

!kaggle --version || true

## 5. Download SLICE-3D / ISIC 2024 data

This uses the Kaggle competition path. You must accept the competition rules on Kaggle first.

If Kaggle download is too large, start with:
- metadata only,
- a small subset of images,
- or the official ISIC Challenge dataset download page.

The repo expects a CSV with at least:
`image_path, patient_id, lesion_id, label`.

In [ ]:
DATA_DIR = "/content/data/isic2024"
os.makedirs(DATA_DIR, exist_ok=True)

# Full Kaggle competition download. This may be large.
# Uncomment after Kaggle API is configured and competition rules are accepted.
# !kaggle competitions download -c isic-2024-challenge -p {DATA_DIR}
# !unzip -n {DATA_DIR}/isic-2024-challenge.zip -d {DATA_DIR}

print("Data dir:", DATA_DIR)
!find {DATA_DIR} -maxdepth 2 -type f | head -20 || true

## 5B. If Kaggle provides HDF5 images, export a subset to JPEG

In [ ]:
# ISIC 2024 Kaggle distributions often include train-image.hdf5.
# This helper exports a manageable subset so the repo can use image_path-based CSVs.

import os, h5py, pandas as pd
from PIL import Image
import io, tqdm, numpy as np

metadata_path = os.path.join(DATA_DIR, "train-metadata.csv")
hdf5_path = os.path.join(DATA_DIR, "train-image.hdf5")
export_dir = os.path.join(DATA_DIR, "train_images_jpg")
os.makedirs(export_dir, exist_ok=True)

MAX_IMAGES = 5000  # adjust for Colab storage/time

if os.path.exists(metadata_path) and os.path.exists(hdf5_path):
    meta = pd.read_csv(metadata_path)
    print("Metadata rows:", len(meta), "columns:", meta.columns.tolist()[:20])
    # Prioritize patients with enough lesions and include positives if possible.
    if "target" in meta.columns:
        meta_sorted = pd.concat([
            meta[meta["target"] == 1],
            meta[meta["target"] == 0]
        ]).drop_duplicates()
    else:
        meta_sorted = meta
    subset = meta_sorted.head(MAX_IMAGES).copy()

    with h5py.File(hdf5_path, "r") as h5:
        keys = set(h5.keys())
        for isic_id in tqdm.tqdm(subset["isic_id"].astype(str), total=len(subset)):
            out_path = os.path.join(export_dir, f"{isic_id}.jpg")
            if os.path.exists(out_path):
                continue
            if isic_id not in keys:
                continue
            img_bytes = h5[isic_id][()]
            img = Image.open(io.BytesIO(img_bytes))
            img.save(out_path, quality=95)

    print("Exported jpg count:", len(glob.glob(export_dir + "/*.jpg")))
else:
    print("No metadata+hdf5 found yet. Download/unzip data first.")

## 6. Prepare SLICE-3D CSV for Experiment 1

In [ ]:
%cd {WORKDIR}

metadata_path = os.path.join(DATA_DIR, "train-metadata.csv")
image_root = os.path.join(DATA_DIR, "train_images_jpg")
out_csv = "outputs/slice3d_ready.csv"

if os.path.exists(metadata_path) and os.path.exists(image_root):
    !python scripts/prepare_slice3d.py \
        --input {metadata_path} \
        --image-root {image_root} \
        --output {out_csv}
    !python scripts/verify_splits.py --csv {out_csv}
else:
    print("SLICE-3D metadata/images not found yet.")

## 7. Run real SLICE-3D reranking experiment

In [ ]:
%cd {WORKDIR}
out_csv = "outputs/slice3d_ready.csv"

# Start with a manageable subset in the script/config if needed.
# The key requirement is patient-disjoint splitting.
if os.path.exists(out_csv):
    !python run_experiment0.py --csv {out_csv}
else:
    print("No prepared SLICE-3D CSV yet.")

## 8. Detector diagnostic branch: toy first-stage candidate generation

This runs Experiment 0.5:
`image -> high-recall candidate boxes -> TP/FP/missed analysis -> candidate_crops.csv`.

This is pipeline validation before real iToBoS.

In [ ]:
%cd {WORKDIR}

# Run toy detector diagnostic.
!python scripts/run_detector.py --config configs/detector_improved.yaml || python scripts/run_detector.py

print("Detector outputs:")
!find detector_outputs -maxdepth 2 -type f | head -50 || true

## 9. Download iToBoS detection data

Use this after accepting the Kaggle competition/data terms.

The expected real detector CSV schema is:
`image_path, image_id, bbox_x1, bbox_y1, bbox_x2, bbox_y2, class_id, patient_id, split`.

In [ ]:
ITOBOS_DIR = "/content/data/itobos"
os.makedirs(ITOBOS_DIR, exist_ok=True)

# Uncomment after Kaggle API is configured and access is accepted.
# !kaggle competitions download -c itobos-2024-detection -p {ITOBOS_DIR}
# !unzip -n {ITOBOS_DIR}/itobos-2024-detection.zip -d {ITOBOS_DIR}

print("iToBoS dir:", ITOBOS_DIR)
!find {ITOBOS_DIR} -maxdepth 2 -type f | head -30 || true

## 10. Run detector on real iToBoS-style annotations

In [ ]:
%cd {WORKDIR}

# After converting iToBoS annotations to the required CSV schema:
ITOBOS_CSV = "/content/data/itobos/itobos_annotations_ready.csv"

if os.path.exists(ITOBOS_CSV):
    !python scripts/run_detector.py --csv {ITOBOS_CSV} --config configs/detector_improved.yaml
    # Then bridge to Stage 2:
    if os.path.exists("detector_outputs/candidate_crops.csv"):
        !python run_experiment0.py --csv detector_outputs/candidate_crops.csv
else:
    print("Prepare iToBoS CSV first:", ITOBOS_CSV)

## 11. Zip outputs for download

In [ ]:
%cd {WORKDIR}

!zip -r /content/thesis_outputs.zip outputs detector_outputs configs src scripts tests README.md requirements.txt run_experiment0.py >/dev/null 2>&1 || true
print("Created /content/thesis_outputs.zip")
# from google.colab import files
# files.download("/content/thesis_outputs.zip")